**Notebook intro**
The RGB and IR streams appear to be mounted on the same moving base, so this notebook explores how to geometrically align the two modalities without leaking test-set labels.


Purpose of this notebook: Check possibilties of aligning two datastreams, they seem to move and focus on the exact same area (probably on a single moving base). How can we make their vision align as good as possible without data leakage (cant use labels of test set in our final algorithm)

**Load one frame per modality & naive overlay** — Reads frame 0 from `visible.mp4` and `infrared.mp4` with OpenCV (`cv2.VideoCapture`), prints both shapes, then overlays them at 50% opacity after simply resizing the IR frame to the RGB resolution (`cv2.resize`). Serves as a baseline showing the raw misalignment before any correction.


In [ ]:
"""Plot overlapping frames of both modalities (50% opacity) to visualize the alignment of the two modalities."""
from pathlib import Path

import cv2
import json
import matplotlib.pyplot as plt
import numpy as np

dataset_path = Path("../data/Anti-UAV-RGBT")
seq = dataset_path / "test" / "20190925_111757_1_1"
frame_id = 0


def read_frame(video_path: Path, frame_id: int = 0) -> np.ndarray:
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise ValueError(f"could not read frame {frame_id} from {video_path}")
    return frame


rgb = read_frame(seq / "visible.mp4", frame_id)
ir = read_frame(seq / "infrared.mp4", frame_id)
print(f"rgb: {rgb.shape}, ir: {ir.shape}")

# naive overlay: resize the ir frame to the rgb frame size
ir_resized = cv2.resize(ir, (rgb.shape[1], rgb.shape[0]))

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB), alpha=0.5)
plt.imshow(cv2.cvtColor(ir_resized, cv2.COLOR_BGR2RGB), alpha=0.5)
plt.title("raw overlay (ir resized to rgb size)")
plt.axis("off")
plt.show()

**Single-frame alignment via GT box centers** — Loads the ground-truth boxes from `visible.json` / `infrared.json`, computes each box's center (`x+w/2, y+h/2`), and derives an affine transform (scale = rgb_size/ir_size, translation so both centers coincide). The IR frame is warped into the RGB canvas with `cv2.warpAffine` and both frames are overlaid at 50% opacity to visually check alignment on this one frame.


In [ ]:
"""Plot overlapping frames of both modalities (50% opacity) aligned on the center of the UAV GT box to visualize the alignment of the two modalities."""
gt_rgb = json.loads((seq / "visible.json").read_text())
gt_ir = json.loads((seq / "infrared.json").read_text())


def box_center(rect) -> tuple[float, float]:
    x, y, w, h = rect
    return x + w / 2, y + h / 2


rgb_center = box_center(gt_rgb["gt_rect"][frame_id])
ir_center = box_center(gt_ir["gt_rect"][frame_id])

# scale the ir frame onto the rgb canvas and translate it
# so that both gt box centers coincide
scale = (rgb.shape[1] / ir.shape[1], rgb.shape[0] / ir.shape[0])
offset = (
    rgb_center[0] - scale[0] * ir_center[0],
    rgb_center[1] - scale[1] * ir_center[1],
)
M = np.array([[scale[0], 0, offset[0]], [0, scale[1], offset[1]]], dtype=np.float32)
ir_centered = cv2.warpAffine(ir, M, (rgb.shape[1], rgb.shape[0]))

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB), alpha=0.5)
plt.imshow(cv2.cvtColor(ir_centered, cv2.COLOR_BGR2RGB), alpha=0.5)
plt.title(f"overlay aligned on gt box center {rgb_center}")
plt.axis("off")
plt.show()

**Learn a global alignment (scale + offset) by regression on train-split GT** — Collects GT box centers from all *train* sequences (test labels are never touched → no data leakage), fits a per-axis `sklearn.LinearRegression` mapping IR centers → RGB centers, and reports scale, offset and R². It plots the regression lines over the per-sequence scatter, then evaluates MAE / RMSE / R² on both the train (fit) and val (held-out) splits to check generalization.


In [ ]:
"""Use many UAV ground thruth boxes to calculate a regression based offset and scale factor to align the two modalities."""
from sklearn.linear_model import LinearRegression

# only train split labels are used for the regression (no test labels -> no data leakage)


def gt_centers(seq_dir: Path) -> tuple[np.ndarray, np.ndarray]:
    out = []
    for mod in ("visible", "infrared"):
        gt = json.loads((seq_dir / f"{mod}.json").read_text())
        rects = [r for r, e in zip(gt["gt_rect"], gt["exist"]) if e == 1 and len(r) == 4]
        if rects:
            arr = np.array(rects, dtype=float)
            out.append(arr[:, :2] + arr[:, 2:] / 2)  # (x, y, w, h) -> center (x, y)
        else:
            out.append(np.empty((0, 2)))
    n = min(len(out[0]), len(out[1]))
    return out[0][:n], out[1][:n]


rgb_centers, ir_centers = [], []
for seq_dir in sorted((dataset_path / "train").iterdir()):
    if not seq_dir.is_dir():
        continue
    c_rgb, c_ir = gt_centers(seq_dir)
    if len(c_rgb):
        rgb_centers.append(c_rgb)
        ir_centers.append(c_ir)

centers_rgb = np.vstack(rgb_centers)
centers_ir = np.vstack(ir_centers)
print(f"{len(centers_rgb)} gt boxes from {len(rgb_centers)} train sequences")

# regress rgb box centers on ir box centers per axis: rgb = scale * ir + offset
reg_x = LinearRegression().fit(centers_ir[:, [0]], centers_rgb[:, 0])
reg_y = LinearRegression().fit(centers_ir[:, [1]], centers_rgb[:, 1])

scale_x, offset_x = float(reg_x.coef_[0]), float(reg_x.intercept_)
scale_y, offset_y = float(reg_y.coef_[0]), float(reg_y.intercept_)

print(f"scale factor: ({scale_x:.4f}, {scale_y:.4f})")
print(f"offset:       ({offset_x:.2f}, {offset_y:.2f})")
r2_x = reg_x.score(centers_ir[:, [0]], centers_rgb[:, 0])
r2_y = reg_y.score(centers_ir[:, [1]], centers_rgb[:, 1])
print(f"R^2:          ({r2_x:.4f}, {r2_y:.4f})")

# plot the regression lines over the gt box centers used for the fit, coloured per sequence
import distinctipy

seq_colors = distinctipy.get_colors(len(rgb_centers), exclude_colors=[(1.0, 0.0, 0.0)])
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, i, name, reg in zip(axes, [0, 1], ["x", "y"], [reg_x, reg_y]):
    for seq_ir, seq_rgb, color in zip(ir_centers, rgb_centers, seq_colors):
        ax.scatter(seq_ir[:, i], seq_rgb[:, i], s=1, alpha=0.5, color=color)
    ir_axis = np.linspace(centers_ir[:, i].min(), centers_ir[:, i].max(), 100)
    ax.plot(ir_axis, reg.predict(ir_axis[:, None]), color="red", label="regression line")
    ax.set_xlabel(f"ir {name} center")
    ax.set_ylabel(f"rgb {name} center")
    ax.legend()
plt.suptitle("regression lines: rgb gt box center on ir gt box center")
plt.tight_layout()
plt.show()

# error metrics of the regression: train (fit) and val (held-out, still no test labels)
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error


def regression_metrics(c_ir: np.ndarray, c_rgb: np.ndarray) -> dict[str, tuple[float, float]]:
    pred_x = reg_x.predict(c_ir[:, [0]])
    pred_y = reg_y.predict(c_ir[:, [1]])
    return {
        "MAE (px)": (
            mean_absolute_error(c_rgb[:, 0], pred_x),
            mean_absolute_error(c_rgb[:, 1], pred_y),
        ),
        "RMSE (px)": (
            root_mean_squared_error(c_rgb[:, 0], pred_x),
            root_mean_squared_error(c_rgb[:, 1], pred_y),
        ),
        "R^2": (r2_score(c_rgb[:, 0], pred_x), r2_score(c_rgb[:, 1], pred_y)),
    }


val_rgb, val_ir = [], []
for seq_dir in sorted((dataset_path / "val").iterdir()):
    if not seq_dir.is_dir():
        continue
    c_rgb, c_ir = gt_centers(seq_dir)
    if len(c_rgb):
        val_rgb.append(c_rgb)
        val_ir.append(c_ir)
val_centers_rgb = np.vstack(val_rgb)
val_centers_ir = np.vstack(val_ir)

for name, c_ir, c_rgb in [
    ("train", centers_ir, centers_rgb),
    ("val", val_centers_ir, val_centers_rgb),
]:
    print(f"{name} ({len(c_rgb)} gt boxes):")
    for metric, (m_x, m_y) in regression_metrics(c_ir, c_rgb).items():
        print(f"  {metric}: x={m_x:.4f}, y={m_y:.4f}")

**Apply the learned transform & visualize the result** — Builds the affine matrix from the regression's scale/offset (`rgb = scale * ir + offset`), warps the IR frame into the RGB coordinate system with `cv2.warpAffine`, and plots the 50%-opacity overlay so the alignment quality of the learned global transform can be judged visually.


In [ ]:
"""Plot both modalities (50% opacity)according to the regression based offset and scale factor to visualize the alignment of the two modalities."""
# warp the ir frame into the rgb frame's coordinate system
# using the regression based scale factor and offset: rgb = scale * ir + offset
M = np.array([[scale_x, 0, offset_x], [0, scale_y, offset_y]], dtype=np.float32)
ir_aligned = cv2.warpAffine(ir, M, (rgb.shape[1], rgb.shape[0]))

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB), alpha=0.5)
plt.imshow(cv2.cvtColor(ir_aligned, cv2.COLOR_BGR2RGB), alpha=0.5)
plt.title(
    f"overlay aligned by regression (scale=({scale_x:.3f}, {scale_y:.3f}), "
    f"offset=({offset_x:.1f}, {offset_y:.1f}))"
)
plt.axis("off")
plt.show()